<a href="https://colab.research.google.com/github/RaziehZare/Speech-Processing-Ontology/blob/main/02ProtocolIII.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
# X-Vector Speaker Verification Evaluation Notebook
Comprehensive evaluation of x-vector model performance on verification trials, aligned with the training implementation
duration of each sound 1 to 2 sec
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import glob
from scipy.optimize import brentq
from scipy import interpolate
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_recall_curve, average_precision_score
from sklearn.metrics import det_curve, confusion_matrix
import seaborn as sns
from tqdm import tqdm
import random

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define paths
base_dir = "/content/drive/MyDrive"
model_path = os.path.join(base_dir, "Speaker-Recognition-x-vectors/checkpoints/xvector_verification_epoch_50.pth")
trials_dir = os.path.join(base_dir, "VerificationTrialsV2")

# 1. Model Architecture (Same as in training notebook)

class TDNN(nn.Module):
    """
    TDNN (Time Delay Neural Network) layer
    """
    def __init__(self, input_dim=None, output_dim=None, context_size=None, dilation=None, dropout_p=0.0):
        super(TDNN, self).__init__()
        self.context_size = context_size
        self.dilation = dilation

        self.kernel = nn.Conv1d(input_dim, output_dim, context_size, dilation=dilation)
        self.nonlinearity = nn.ReLU()
        self.bn = nn.BatchNorm1d(output_dim)
        self.dropout = nn.Dropout(p=dropout_p)

    def forward(self, x):
        """
        x: [batch, seq_len, input_dim]
        output: [batch, new_seq_len, output_dim]
        """
        # Convert to [batch, input_dim, seq_len] for Conv1d
        x = x.transpose(1, 2)

        # Apply convolution, nonlinearity, and batch normalization
        x = self.kernel(x)
        x = self.nonlinearity(x)
        x = self.bn(x)

        # Apply dropout
        x = self.dropout(x)

        # Convert back to [batch, seq_len, output_dim]
        x = x.transpose(1, 2)

        return x

class StatisticsPooling(nn.Module):
    """
    Statistics pooling layer that computes mean and standard deviation
    """
    def __init__(self):
        super(StatisticsPooling, self).__init__()

    def forward(self, x):
        # x: [batch, seq_len, hidden_dim]

        # Calculate mean and standard deviation along the time dimension
        mean = torch.mean(x, dim=1)
        std = torch.std(x, dim=1)

        # Concatenate mean and standard deviation
        pooled = torch.cat((mean, std), dim=1)

        return pooled

class XVectorTDNN(nn.Module):
    """
    X-vector TDNN architecture for speaker verification
    """
    def __init__(self, input_dim=40, embedding_dim=512):
        super(XVectorTDNN, self).__init__()

        # Frame-level layers
        self.frame_layers = nn.Sequential(
            TDNN(input_dim=input_dim, output_dim=512, context_size=5, dilation=1, dropout_p=0.3),
            TDNN(input_dim=512, output_dim=512, context_size=3, dilation=2, dropout_p=0.3),
            TDNN(input_dim=512, output_dim=512, context_size=3, dilation=3, dropout_p=0.3),
            TDNN(input_dim=512, output_dim=512, context_size=1, dilation=1, dropout_p=0.3),
            TDNN(input_dim=512, output_dim=1500, context_size=1, dilation=1, dropout_p=0.3)
        )

        # Statistics pooling layer
        self.stats_pooling = StatisticsPooling()

        # Segment-level layers
        self.segment_layer1 = nn.Sequential(
            nn.Linear(3000, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3)
        )

        # Embedding layer (x-vector)
        self.embedding_layer = nn.Sequential(
            nn.Linear(512, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )

    def forward(self, x, return_embedding=True):
        # Input: (batch, time, feature)

        # Frame-level processing
        x = self.frame_layers(x)

        # Statistics pooling
        x = self.stats_pooling(x)

        # Segment-level processing
        x = self.segment_layer1(x)

        # Embedding layer (x-vector)
        embeddings = self.embedding_layer(x)

        return embeddings

# 2. Feature Extraction (Same as in training notebook)

def extract_features(audio_path, n_mels=40):
    waveform, sample_rate = torchaudio.load(audio_path)

    # Ensure mono audio
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Resample if needed
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
        sample_rate = 16000

    # Extract Mel-filterbank features (as in training)
    melspec_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate,
        n_fft=400,  # 25ms window
        hop_length=160,  # 10ms hop
        n_mels=n_mels,
        power=2.0
    )

    # Log mel spectrograms
    melspec = melspec_transform(waveform)
    log_melspec = torch.log(melspec + 1e-9)

    # Normalize along frequency axis (mean and std computed across time)
    mean = torch.mean(log_melspec, dim=2, keepdim=True)
    std = torch.std(log_melspec, dim=2, keepdim=True) + 1e-9
    normalized = (log_melspec - mean) / std

    return normalized.squeeze(0).transpose(0, 1)  # (time, feature)

# 3. Helper functions for evaluation

# Extract speaker ID from filename
def extract_speaker_id(filename):
    # Example: spk028-m-cl-cutpauses-normalized-smpl5.wav
    return int(filename.split('-')[0][3:])

# Extract speech style from filename
def extract_style(filename):
    # Example: spk028-m-cl-cutpauses-normalized-smpl5.wav
    style_part = filename.split('-')[2]
    return style_part  # 'cl', 'ch', or 'rd'

# Compute cosine similarity scores between embeddings
def compute_cosine_scores(enrollment_embeds, test_embeds):
    """
    Compute cosine similarity scores between enrollment and test embeddings
    Corrected to handle dimensions properly
    """
    # Ensure tensors are properly shaped
    if len(enrollment_embeds.shape) == 3:
        enrollment_embeds = enrollment_embeds.squeeze(0)
    if len(test_embeds.shape) == 3:
        test_embeds = test_embeds.squeeze(0)

    # Normalize embeddings
    enrollment_embeds = F.normalize(enrollment_embeds, p=2, dim=1)
    test_embeds = F.normalize(test_embeds, p=2, dim=1)

    # Compute cosine similarity
    scores = torch.mm(test_embeds, enrollment_embeds.t())

    return scores

# MISSING FUNCTION - Detection Cost Function (DCF)
def compute_dcf(labels, scores, p_target=0.01, c_miss=1, c_fa=1):
    """
    Compute Detection Cost Function (DCF) for speaker verification

    Args:
        labels: Binary labels (1 for target/same speaker, 0 for non-target/different speaker)
        scores: Similarity scores (higher = more similar)
        p_target: Prior probability of target trials
        c_miss: Cost of miss (false rejection)
        c_fa: Cost of false alarm (false acceptance)

    Returns:
        min_dcf: Minimum Detection Cost Function
        threshold: Threshold that achieves minimum DCF
    """
    # Ensure numpy arrays
    if isinstance(scores, torch.Tensor):
        scores = scores.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    # Compute ROC curve
    fpr, tpr, thresholds = roc_curve(labels, scores)

    # Compute miss rate (false negative rate)
    fnr = 1 - tpr

    # Compute DCF for each threshold
    dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)

    # Find minimum DCF
    min_dcf_idx = np.argmin(dcf)
    min_dcf = dcf[min_dcf_idx]
    threshold = thresholds[min_dcf_idx]

    return min_dcf, threshold

# Improved EER computation function with better error handling
def compute_eer_improved(labels, scores):
    """
    Compute Equal Error Rate (EER) with improved robustness

    Args:
        labels: Binary labels where 1 = same speaker, 0 = different speaker
        scores: Similarity scores (higher = more similar)

    Returns:
        eer: Equal Error Rate
        threshold: Threshold at EER
        fpr: False Positive Rate array
        fnr: False Negative Rate array
        thresholds: Threshold array
    """
    # Ensure numpy arrays
    if isinstance(scores, torch.Tensor):
        scores = scores.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    # Debug info
    print(f"EER computation - labels shape: {labels.shape}, unique values: {np.unique(labels, return_counts=True)}")
    print(f"EER computation - scores shape: {scores.shape}, range: [{np.min(scores)}, {np.max(scores)}]")

    # Check for invalid values
    if np.any(np.isnan(scores)) or np.any(np.isinf(scores)):
        print("Warning: NaN or Inf values found in scores, removing them...")
        valid_mask = np.isfinite(scores)
        scores = scores[valid_mask]
        labels = labels[valid_mask]

    # Compute ROC curve
    try:
        fpr, tpr, thresholds = roc_curve(labels, scores)
    except Exception as e:
        print(f"Error computing ROC curve: {e}")
        return 0.5, 0.0, np.array([0.5]), np.array([0.5]), np.array([0.0])

    # Compute False Negative Rate (False Rejection Rate)
    fnr = 1 - tpr

    # Find the point where FPR and FNR are closest (EER point)
    # Method 1: Direct minimization of |FPR - FNR|
    diff = np.abs(fpr - fnr)
    eer_idx = np.argmin(diff)
    eer_direct = (fpr[eer_idx] + fnr[eer_idx]) / 2
    threshold_direct = thresholds[eer_idx]

    # Method 2: Interpolation method (more accurate) with better error handling
    try:
        # Create interpolation functions
        if len(np.unique(fpr)) > 1 and len(np.unique(fnr)) > 1:
            # Sort by threshold for proper interpolation
            sorted_indices = np.argsort(thresholds)
            fpr_sorted = fpr[sorted_indices]
            fnr_sorted = fnr[sorted_indices]
            thresholds_sorted = thresholds[sorted_indices]

            # Check if we have enough unique values for interpolation
            if len(np.unique(thresholds_sorted)) < 3:
                print("Not enough unique threshold values for interpolation, using direct method")
                return eer_direct, threshold_direct, fpr, fnr, thresholds

            # Find intersection using interpolation
            from scipy.interpolate import interp1d

            # Create a fine grid of threshold values
            thresh_min, thresh_max = thresholds_sorted.min(), thresholds_sorted.max()
            if thresh_min == thresh_max:
                print("All thresholds are the same, using direct method")
                return eer_direct, threshold_direct, fpr, fnr, thresholds

            thresh_interp = np.linspace(thresh_min, thresh_max, 10000)

            # Interpolate FPR and FNR with bounds checking
            try:
                fpr_interp_func = interp1d(thresholds_sorted, fpr_sorted, kind='linear',
                                         bounds_error=False, fill_value='extrapolate')
                fnr_interp_func = interp1d(thresholds_sorted, fnr_sorted, kind='linear',
                                         bounds_error=False, fill_value='extrapolate')

                fpr_interp = fpr_interp_func(thresh_interp)
                fnr_interp = fnr_interp_func(thresh_interp)

                # Check for NaN values in interpolation
                if np.any(np.isnan(fpr_interp)) or np.any(np.isnan(fnr_interp)):
                    print("NaN values in interpolation, using direct method")
                    return eer_direct, threshold_direct, fpr, fnr, thresholds

                # Find minimum difference
                diff_interp = np.abs(fpr_interp - fnr_interp)
                min_idx = np.argmin(diff_interp)

                eer_interp = (fpr_interp[min_idx] + fnr_interp[min_idx]) / 2
                threshold_interp = thresh_interp[min_idx]

                print(f"Direct method EER: {eer_direct:.4f}, Threshold: {threshold_direct:.4f}")
                print(f"Interpolation method EER: {eer_interp:.4f}, Threshold: {threshold_interp:.4f}")

                # Use interpolation result if reasonable, otherwise use direct
                if not np.isnan(eer_interp) and abs(eer_interp - eer_direct) < 0.1:  # If results are close
                    return eer_interp, threshold_interp, fpr, fnr, thresholds
                else:
                    print("Large difference between methods or NaN result, using direct method")
                    return eer_direct, threshold_direct, fpr, fnr, thresholds

            except Exception as interp_e:
                print(f"Interpolation failed: {interp_e}")
                return eer_direct, threshold_direct, fpr, fnr, thresholds

    except Exception as e:
        print(f"Interpolation method failed: {e}")

    print(f"Using direct method EER: {eer_direct:.4f}, Threshold: {threshold_direct:.4f}")
    return eer_direct, threshold_direct, fpr, fnr, thresholds

# 4. Process verification trials

# Process a single trial
def process_trial(trial_path, model, device, feature_cache=None):
    if feature_cache is None:
        feature_cache = {}

    # Get the two audio files in the trial
    files = [f for f in os.listdir(trial_path) if f.endswith('.wav')]

    if len(files) != 2:
        print(f"Error: Expected 2 files in {trial_path}, found {len(files)}")
        return None

    # Extract features and embeddings for both files
    embeddings = []
    for file in files:
        file_path = os.path.join(trial_path, file)

        # Check if features are cached
        if file_path in feature_cache:
            features = feature_cache[file_path]
        else:
            try:
                features = extract_features(file_path)
                feature_cache[file_path] = features
            except Exception as e:
                print(f"Error extracting features from {file_path}: {e}")
                return None

        # Extract embedding
        try:
            with torch.no_grad():
                # Add batch dimension
                features_tensor = features.unsqueeze(0).to(device)
                embedding = model(features_tensor)
                embeddings.append(embedding.cpu())
        except Exception as e:
            print(f"Error computing embedding for {file_path}: {e}")
            return None

    # Compute cosine similarity - corrected to ensure proper dimensions
    try:
        embedding1 = embeddings[0]
        embedding2 = embeddings[1]
        similarity = F.cosine_similarity(embedding1, embedding2).item()
    except Exception as e:
        print(f"Error computing similarity: {e}")
        return None

    # Extract metadata
    try:
        speaker1 = extract_speaker_id(files[0])
        speaker2 = extract_speaker_id(files[1])
        style1 = extract_style(files[0])
        style2 = extract_style(files[1])
    except Exception as e:
        print(f"Error extracting metadata from {files}: {e}")
        return None

    is_same_speaker = speaker1 == speaker2

    return {
        'file1': files[0],
        'file2': files[1],
        'speaker1': speaker1,
        'speaker2': speaker2,
        'style1': style1,
        'style2': style2,
        'is_same_speaker': is_same_speaker,
        'pattern': f"{style1}-{style2}",
        'similarity': similarity
    }

# Process all verification trials
def process_verification_trials(model, device):
    results = []
    feature_cache = {}  # Cache to avoid re-extracting features

    # Get all trial folders
    trial_folders = [d for d in os.listdir(trials_dir) if d.startswith("trial")]

    for trial_folder in tqdm(sorted(trial_folders), desc="Processing Verification Trials"):
        trial_path = os.path.join(trials_dir, trial_folder)

        if not os.path.isdir(trial_path):
            continue

        result = process_trial(trial_path, model, device, feature_cache)

        if result:
            # Extract trial number if available
            if "_" in trial_folder:
                trial_num = int(trial_folder.split('_')[1])
                result['trial_num'] = trial_num
            else:
                result['trial_num'] = len(results) + 1

            results.append(result)

    return results

from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve

def compute_cllr_calibrated(y_true, y_score):
    """
    Compute calibrated Cllr using logistic regression calibration
    """
    try:
        y_true = np.array(y_true)
        y_score = np.array(y_score).reshape(-1, 1)  # sklearn expects 2D array

        # Check for constant scores
        if np.var(y_score) == 0:
            print("Warning: All scores are identical, cannot compute Cllr")
            return float('inf')

        # Train logistic regression for calibration
        clf = LogisticRegression(solver='liblinear', max_iter=1000)
        clf.fit(y_score, y_true)

        # Get calibrated probabilities
        probs = clf.predict_proba(y_score)[:, 1]  # P(target=1)

        # Avoid log(0)
        eps = 1e-15
        probs = np.clip(probs, eps, 1 - eps)

        # Compute log-likelihood ratios
        llr = np.log(probs / (1 - probs))

        # Compute Cllr
        llr_target = llr[y_true == 1]
        llr_non_target = llr[y_true == 0]

        if len(llr_target) == 0 or len(llr_non_target) == 0:
            print("Warning: No target or non-target trials, cannot compute Cllr")
            return float('inf')

        cllr_target = np.mean(np.log2(1 + 1 / np.exp(llr_target)))
        cllr_non_target = np.mean(np.log2(1 + np.exp(llr_non_target)))

        cllr = 0.5 * (cllr_target + cllr_non_target)
        return cllr
    except Exception as e:
        print(f"Error computing Cllr: {e}")
        return float('inf')

# 5. Performance Metrics Computation

# Compute comprehensive performance metrics
def compute_verification_metrics(results):
    # Extract ground truth and scores
    y_true = np.array([1 if r['is_same_speaker'] else 0 for r in results])
    y_score = np.array([r['similarity'] for r in results])

    # Print debugging information
    print(f"y_true shape: {y_true.shape}, unique values: {np.unique(y_true, return_counts=True)}")
    print(f"y_score shape: {y_score.shape}, range: [{np.min(y_score)}, {np.max(y_score)}]")

    # Check for valid data
    if len(y_true) == 0 or len(y_score) == 0:
        raise ValueError("No valid trials found")

    if len(np.unique(y_true)) < 2:
        print("Warning: Only one class found in labels")
        # Create dummy metrics
        return create_dummy_metrics(y_true, y_score)

    # Compute EER
    eer, eer_threshold, fpr_eer, fnr_eer, thresholds_eer = compute_eer_improved(y_true, y_score)

    # Compute DCF with different target priors
    try:
        dcf_01, dcf_01_threshold = compute_dcf(y_true, y_score, p_target=0.01)
        dcf_05, dcf_05_threshold = compute_dcf(y_true, y_score, p_target=0.5)
    except Exception as e:
        print(f"Error computing DCF: {e}")
        dcf_01, dcf_01_threshold = 1.0, 0.0
        dcf_05, dcf_05_threshold = 1.0, 0.0

    # Compute predictions at EER threshold
    y_pred = np.array([1 if score >= eer_threshold else 0 for score in y_score])

    # Compute confusion matrix values
    try:
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
        else:
            # Handle case where only one class is predicted
            if np.all(y_pred == 0):  # All predicted as 0
                tn = np.sum(y_true == 0)
                fp = 0
                fn = np.sum(y_true == 1)
                tp = 0
            else:  # All predicted as 1
                tn = 0
                fp = np.sum(y_true == 0)
                fn = 0
                tp = np.sum(y_true == 1)
    except Exception as e:
        print(f"Error computing confusion matrix: {e}")
        tn, fp, fn, tp = 0, 0, 0, 0

    # Calculate metrics
    try:
        accuracy = accuracy_score(y_true, y_pred)
        auc = roc_auc_score(y_true, y_score)
        ap = average_precision_score(y_true, y_score)  # Average precision (area under PR curve)
    except Exception as e:
        print(f"Error computing sklearn metrics: {e}")
        accuracy, auc, ap = 0.0, 0.5, 0.5

    # Calculate FAR and FRR
    far = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Acceptance Rate
    frr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Rejection Rate

    # Get DET curve points
    try:
        fpr, fnr, _ = det_curve(y_true, y_score)
    except Exception as e:
        print(f"Error computing DET curve: {e}")
        fpr, fnr = np.array([0, 1]), np.array([1, 0])

    # Compute Cllr
    cllr = compute_cllr_calibrated(y_true, y_score)

    return {
        'eer': eer,
        'eer_threshold': eer_threshold,
        'fpr_eer': fpr_eer,
        'fnr_eer': fnr_eer,
        'dcf_01': dcf_01,
        'dcf_01_threshold': dcf_01_threshold,
        'dcf_05': dcf_05,
        'dcf_05_threshold': dcf_05_threshold,
        'accuracy': accuracy,
        'auc': auc,
        'ap': ap,
        'hits': tp,  # True Positives
        'misses': fn,  # False Negatives
        'false_alarms': fp,  # False Positives
        'correct_rejections': tn,  # True Negatives
        'hit_rate': tp / (tp + fn) if (tp + fn) > 0 else 0,  # TPR / Recall
        'false_alarm_rate': fp / (fp + tn) if (fp + tn) > 0 else 0,  # FPR
        'far': far,
        'frr': frr,
        'fpr': fpr,  # For DET curve
        'fnr': fnr,  # For DET curve
        'y_true': y_true,
        'y_score': y_score,
        'y_pred': y_pred,
        'cllr': cllr,
    }

def create_dummy_metrics(y_true, y_score):
    """Create dummy metrics when computation fails"""
    return {
        'eer': 0.5,
        'eer_threshold': 0.0,
        'fpr_eer': np.array([0.5]),
        'fnr_eer': np.array([0.5]),
        'dcf_01': 1.0,
        'dcf_01_threshold': 0.0,
        'dcf_05': 1.0,
        'dcf_05_threshold': 0.0,
        'accuracy': 0.5,
        'auc': 0.5,
        'ap': 0.5,
        'hits': 0,
        'misses': 0,
        'false_alarms': 0,
        'correct_rejections': 0,
        'hit_rate': 0.0,
        'false_alarm_rate': 0.0,
        'far': 0.0,
        'frr': 0.0,
        'fpr': np.array([0, 1]),
        'fnr': np.array([1, 0]),
        'y_true': y_true,
        'y_score': y_score,
        'y_pred': np.zeros_like(y_true),
        'cllr': float('inf'),
    }

# Analyze performance by pattern
def analyze_by_pattern(results):
    # Get unique patterns
    patterns = sorted(list(set(r['pattern'] for r in results)))

    pattern_metrics = {}
    for pattern in patterns:
        # Filter results by pattern
        pattern_results = [r for r in results if r['pattern'] == pattern]

        # Skip if too few samples
        if len(pattern_results) < 5:
            print(f"Skipping pattern {pattern}: only {len(pattern_results)} samples")
            continue

        # Compute metrics for this pattern
        try:
            pattern_metrics[pattern] = compute_verification_metrics(pattern_results)
        except Exception as e:
            print(f"Error computing metrics for pattern {pattern}: {e}")
            continue

    return pattern_metrics

# 6. Visualization Functions
# Function to create comprehensive EER plots
def create_comprehensive_eer_plots(all_metrics, pattern_metrics, output_dir, timestamp):
    """
    Create comprehensive EER analysis plots
    """
    # 1. Overall EER plot
    try:
        fig_eer_overall, eer_val, eer_thresh = plot_eer_analysis(
            all_metrics['y_true'],
            all_metrics['y_score'],
            title="Overall EER Analysis - Speaker Verification"
        )
        fig_eer_overall.savefig(
            os.path.join(output_dir, f"eer_analysis_overall_{timestamp}.png"),
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig_eer_overall)
    except Exception as e:
        print(f"Error creating overall EER plot: {e}")

    # 2. EER plots by pattern (subplot)
    n_patterns = len(pattern_metrics)
    if n_patterns > 0:
        try:
            # Calculate subplot layout
            n_cols = min(3, n_patterns)
            n_rows = (n_patterns + n_cols - 1) // n_cols

            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
            if n_patterns == 1:
                axes = [axes]
            elif n_rows == 1:
                axes = axes.reshape(1, -1)

            for idx, (pattern, metrics) in enumerate(pattern_metrics.items()):
                row = idx // n_cols
                col = idx % n_cols
                ax = axes[row, col] if n_rows > 1 else axes[col]

                # Plot for this pattern
                fpr_eer = metrics['fpr_eer']
                fnr_eer = metrics['fnr_eer']
                eer = metrics['eer']

                # Continuing from the visualization functions...

                ax.plot(fpr_eer * 100, fnr_eer * 100, 'b-', linewidth=2, label='ROC curve')
                ax.plot([0, 100], [0, 100], 'r--', linewidth=1, label='Random classifier')

                # Mark EER point
                ax.plot(eer * 100, eer * 100, 'ro', markersize=8, label=f'EER = {eer:.3f}')

                ax.set_xlabel('False Positive Rate (%)')
                ax.set_ylabel('False Negative Rate (%)')
                ax.set_title(f'EER Analysis - {pattern}')
                ax.grid(True, alpha=0.3)
                ax.legend()
                ax.set_xlim([0, 100])
                ax.set_ylim([0, 100])

            # Hide empty subplots
            for idx in range(n_patterns, n_rows * n_cols):
                row = idx // n_cols
                col = idx % n_cols
                ax = axes[row, col] if n_rows > 1 else axes[col]
                ax.set_visible(False)

            plt.tight_layout()
            fig.savefig(
                os.path.join(output_dir, f"eer_analysis_by_pattern_{timestamp}.png"),
                dpi=300, bbox_inches='tight'
            )
            plt.close(fig)
        except Exception as e:
            print(f"Error creating pattern EER plots: {e}")

def plot_eer_analysis(y_true, y_score, title="EER Analysis"):
    """
    Create detailed EER analysis plot
    """
    # Compute ROC curve data
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr

    # Find EER point
    diff = np.abs(fpr - fnr)
    eer_idx = np.argmin(diff)
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
    eer_threshold = thresholds[eer_idx]

    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    # Plot FPR and FNR vs threshold
    ax.plot(thresholds, fpr * 100, 'b-', linewidth=2, label='False Positive Rate')
    ax.plot(thresholds, fnr * 100, 'r-', linewidth=2, label='False Negative Rate')

    # Mark EER point
    ax.plot(eer_threshold, eer * 100, 'go', markersize=10,
            label=f'EER = {eer:.3f} at threshold = {eer_threshold:.3f}')

    ax.set_xlabel('Threshold')
    ax.set_ylabel('Error Rate (%)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()

    return fig, eer, eer_threshold

def plot_det_curve(metrics, title="DET Curve"):
    """
    Create Detection Error Tradeoff (DET) curve
    """
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    # Plot DET curve
    fpr = metrics['fpr'] * 100
    fnr = metrics['fnr'] * 100

    ax.loglog(fpr, fnr, 'b-', linewidth=2, label='DET Curve')
    ax.plot([0.1, 50], [0.1, 50], 'r--', linewidth=1, label='Equal Error Line')

    # Mark EER point
    eer = metrics['eer'] * 100
    ax.plot(eer, eer, 'ro', markersize=8, label=f'EER = {eer:.2f}%')

    ax.set_xlabel('False Alarm Rate (%)')
    ax.set_ylabel('Miss Rate (%)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_xlim([0.1, 50])
    ax.set_ylim([0.1, 50])

    return fig

def plot_score_distribution(results, title="Score Distribution"):
    """
    Plot score distributions for same/different speaker pairs
    """
    # Separate scores by same/different speaker
    same_scores = [r['similarity'] for r in results if r['is_same_speaker']]
    diff_scores = [r['similarity'] for r in results if not r['is_same_speaker']]

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    # Plot histograms
    ax.hist(same_scores, bins=30, alpha=0.7, label=f'Same Speaker (n={len(same_scores)})',
            color='green', density=True)
    ax.hist(diff_scores, bins=30, alpha=0.7, label=f'Different Speaker (n={len(diff_scores)})',
            color='red', density=True)

    ax.set_xlabel('Cosine Similarity Score')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

    return fig

def plot_confusion_matrix(metrics, title="Confusion Matrix"):
    """
    Plot confusion matrix
    """
    cm = confusion_matrix(metrics['y_true'], metrics['y_pred'])

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Different Speaker', 'Same Speaker'],
                yticklabels=['Different Speaker', 'Same Speaker'])

    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)

    return fig

def create_comprehensive_plots(all_metrics, pattern_metrics, results, output_dir, timestamp):
    """
    Create all visualization plots
    """
    print("Creating comprehensive visualization plots...")

    # 1. Overall score distribution
    try:
        fig_dist = plot_score_distribution(results, "Overall Score Distribution")
        fig_dist.savefig(os.path.join(output_dir, f"score_distribution_{timestamp}.png"),
                        dpi=300, bbox_inches='tight')
        plt.close(fig_dist)
    except Exception as e:
        print(f"Error creating score distribution plot: {e}")

    # 2. Overall confusion matrix
    try:
        fig_cm = plot_confusion_matrix(all_metrics, "Overall Confusion Matrix")
        fig_cm.savefig(os.path.join(output_dir, f"confusion_matrix_{timestamp}.png"),
                      dpi=300, bbox_inches='tight')
        plt.close(fig_cm)
    except Exception as e:
        print(f"Error creating confusion matrix plot: {e}")

    # 3. DET curve
    try:
        fig_det = plot_det_curve(all_metrics, "Overall DET Curve")
        fig_det.savefig(os.path.join(output_dir, f"det_curve_{timestamp}.png"),
                       dpi=300, bbox_inches='tight')
        plt.close(fig_det)
    except Exception as e:
        print(f"Error creating DET curve plot: {e}")

    # 4. ROC curve
    try:
        fig_roc, ax_roc = plt.subplots(1, 1, figsize=(8, 6))
        fpr, tpr, _ = roc_curve(all_metrics['y_true'], all_metrics['y_score'])
        auc = all_metrics['auc']

        ax_roc.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {auc:.3f})')
        ax_roc.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier')
        ax_roc.set_xlabel('False Positive Rate')
        ax_roc.set_ylabel('True Positive Rate')
        ax_roc.set_title('ROC Curve - Speaker Verification')
        ax_roc.grid(True, alpha=0.3)
        ax_roc.legend()

        fig_roc.savefig(os.path.join(output_dir, f"roc_curve_{timestamp}.png"),
                       dpi=300, bbox_inches='tight')
        plt.close(fig_roc)
    except Exception as e:
        print(f"Error creating ROC curve plot: {e}")

    # 5. Performance comparison by pattern
    if pattern_metrics:
        try:
            patterns = list(pattern_metrics.keys())
            eers = [pattern_metrics[p]['eer'] * 100 for p in patterns]
            aucs = [pattern_metrics[p]['auc'] for p in patterns]

            fig_comp, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

            # EER comparison
            bars1 = ax1.bar(patterns, eers, color='skyblue', alpha=0.7)
            ax1.set_ylabel('EER (%)')
            ax1.set_title('EER by Speaking Style Pattern')
            ax1.grid(True, alpha=0.3)

            # Add value labels on bars
            for bar, eer in zip(bars1, eers):
                height = bar.get_height()
                ax1.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                        f'{eer:.2f}%', ha='center', va='bottom')

            # AUC comparison
            bars2 = ax2.bar(patterns, aucs, color='lightcoral', alpha=0.7)
            ax2.set_ylabel('AUC')
            ax2.set_title('AUC by Speaking Style Pattern')
            ax2.grid(True, alpha=0.3)
            ax2.set_ylim([0, 1])

            # Add value labels on bars
            for bar, auc in zip(bars2, aucs):
                height = bar.get_height()
                ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{auc:.3f}', ha='center', va='bottom')

            plt.tight_layout()
            fig_comp.savefig(os.path.join(output_dir, f"performance_comparison_{timestamp}.png"),
                           dpi=300, bbox_inches='tight')
            plt.close(fig_comp)
        except Exception as e:
            print(f"Error creating performance comparison plot: {e}")

# 7. Results Analysis and Reporting

def print_detailed_results(all_metrics, pattern_metrics, results):
    """
    Print comprehensive results analysis
    """
    print("\n" + "="*80)
    print("X-VECTOR SPEAKER VERIFICATION EVALUATION RESULTS")
    print("="*80)

    # Overall statistics
    print(f"\nOVERALL STATISTICS:")
    print(f"Total trials: {len(results)}")
    same_speaker_trials = sum(1 for r in results if r['is_same_speaker'])
    diff_speaker_trials = len(results) - same_speaker_trials
    print(f"Same speaker trials: {same_speaker_trials}")
    print(f"Different speaker trials: {diff_speaker_trials}")

    # Overall performance metrics
    print(f"\nOVERALL PERFORMANCE METRICS:")
    print(f"Equal Error Rate (EER): {all_metrics['eer']:.4f} ({all_metrics['eer']*100:.2f}%)")
    print(f"EER Threshold: {all_metrics['eer_threshold']:.4f}")
    print(f"Area Under Curve (AUC): {all_metrics['auc']:.4f}")
    print(f"Average Precision (AP): {all_metrics['ap']:.4f}")
    print(f"Accuracy at EER threshold: {all_metrics['accuracy']:.4f}")
    print(f"Detection Cost Function (DCF) @ p_target=0.01: {all_metrics['dcf_01']:.4f}")
    print(f"Detection Cost Function (DCF) @ p_target=0.5: {all_metrics['dcf_05']:.4f}")
    print(f"Calibrated Log-Likelihood Ratio (Cllr): {all_metrics['cllr']:.4f}")

    # Confusion matrix details
    print(f"\nCONFUSION MATRIX DETAILS:")
    print(f"True Positives (Hits): {all_metrics['hits']}")
    print(f"False Negatives (Misses): {all_metrics['misses']}")
    print(f"False Positives (False Alarms): {all_metrics['false_alarms']}")
    print(f"True Negatives (Correct Rejections): {all_metrics['correct_rejections']}")
    print(f"Hit Rate (True Positive Rate): {all_metrics['hit_rate']:.4f}")
    print(f"False Alarm Rate (False Positive Rate): {all_metrics['false_alarm_rate']:.4f}")
    print(f"False Acceptance Rate (FAR): {all_metrics['far']:.4f}")
    print(f"False Rejection Rate (FRR): {all_metrics['frr']:.4f}")

    # Performance by pattern
    if pattern_metrics:
        print(f"\nPERFORMANCE BY SPEAKING STYLE PATTERN:")
        print(f"{'Pattern':<12} {'Trials':<8} {'EER':<8} {'AUC':<8} {'Accuracy':<10} {'DCF(0.01)':<10}")
        print("-" * 70)

        for pattern, metrics in pattern_metrics.items():
            pattern_trials = sum(1 for r in results if r['pattern'] == pattern)
            print(f"{pattern:<12} {pattern_trials:<8} {metrics['eer']:.3f}    {metrics['auc']:.3f}    "
                  f"{metrics['accuracy']:.3f}      {metrics['dcf_01']:.3f}")

    # Score statistics
    scores = [r['similarity'] for r in results]
    same_scores = [r['similarity'] for r in results if r['is_same_speaker']]
    diff_scores = [r['similarity'] for r in results if not r['is_same_speaker']]

    print(f"\nSCORE STATISTICS:")
    print(f"Overall score range: [{np.min(scores):.4f}, {np.max(scores):.4f}]")
    print(f"Overall score mean: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
    print(f"Same speaker scores: {np.mean(same_scores):.4f} ± {np.std(same_scores):.4f}")
    print(f"Different speaker scores: {np.mean(diff_scores):.4f} ± {np.std(diff_scores):.4f}")
    print(f"Score separation (d'): {(np.mean(same_scores) - np.mean(diff_scores)) / np.sqrt(0.5 * (np.var(same_scores) + np.var(diff_scores))):.4f}")

def save_results_to_files(all_metrics, pattern_metrics, results, output_dir, timestamp):
    """
    Save results to CSV and JSON files
    """
    print("\nSaving results to files...")

    # Save detailed results to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(output_dir, f"verification_results_{timestamp}.csv"), index=False)

    # Save metrics summary to JSON
    metrics_summary = {
        'timestamp': timestamp,
        'overall_metrics': {k: float(v) if isinstance(v, (np.floating, np.integer)) else v
                           for k, v in all_metrics.items()
                           if k not in ['y_true', 'y_score', 'y_pred', 'fpr', 'fnr', 'fpr_eer', 'fnr_eer']},
        'pattern_metrics': {
            pattern: {k: float(v) if isinstance(v, (np.floating, np.integer)) else v
                     for k, v in metrics.items()
                     if k not in ['y_true', 'y_score', 'y_pred', 'fpr', 'fnr', 'fpr_eer', 'fnr_eer']}
            for pattern, metrics in pattern_metrics.items()
        },
        'trial_statistics': {
            'total_trials': len(results),
            'same_speaker_trials': sum(1 for r in results if r['is_same_speaker']),
            'different_speaker_trials': sum(1 for r in results if not r['is_same_speaker'])
        }
    }

    import json
    with open(os.path.join(output_dir, f"metrics_summary_{timestamp}.json"), 'w') as f:
        json.dump(metrics_summary, f, indent=2)

    print(f"Results saved to: {output_dir}")

# 8. Main Evaluation Pipeline

def main_evaluation():
    """
    Main evaluation pipeline
    """
    print("Starting X-Vector Speaker Verification Evaluation...")

    # Create output directory
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = os.path.join(base_dir, f"evaluation_results_{timestamp}")
    os.makedirs(output_dir, exist_ok=True)

    # Load the trained model
    print("Loading trained X-Vector model...")
    try:
        model = XVectorTDNN(input_dim=40, embedding_dim=512)

        # Load checkpoint
        if os.path.exists(model_path):
            checkpoint = torch.load(model_path, map_location=device)
            if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
                print(f"Loaded model from epoch {checkpoint.get('epoch', 'unknown')}")
            else:
                model.load_state_dict(checkpoint)
                print("Loaded model state dict")
        else:
            print(f"Model file not found at {model_path}")
            return

        model.to(device)
        model.eval()
        print("Model loaded successfully!")

    except Exception as e:
        print(f"Error loading model: {e}")
        return

    # Process verification trials
    print("\nProcessing verification trials...")
    try:
        results = process_verification_trials(model, device)
        print(f"Processed {len(results)} verification trials")

        if len(results) == 0:
            print("No valid trials found. Please check the trials directory.")
            return

    except Exception as e:
        print(f"Error processing trials: {e}")
        return

    # Compute overall metrics
    print("\nComputing performance metrics...")
    try:
        all_metrics = compute_verification_metrics(results)
        print("Overall metrics computed successfully!")
    except Exception as e:
        print(f"Error computing overall metrics: {e}")
        return

    # Analyze by pattern
    print("\nAnalyzing performance by speaking style patterns...")
    try:
        pattern_metrics = analyze_by_pattern(results)
        print(f"Analyzed {len(pattern_metrics)} patterns")
    except Exception as e:
        print(f"Error analyzing by pattern: {e}")
        pattern_metrics = {}

    # Create visualizations
    print("\nCreating visualization plots...")
    try:
        create_comprehensive_eer_plots(all_metrics, pattern_metrics, output_dir, timestamp)
        create_comprehensive_plots(all_metrics, pattern_metrics, results, output_dir, timestamp)
        print("Visualization plots created successfully!")
    except Exception as e:
        print(f"Error creating plots: {e}")

    # Print detailed results
    print_detailed_results(all_metrics, pattern_metrics, results)

    # Save results to files
    try:
        save_results_to_files(all_metrics, pattern_metrics, results, output_dir, timestamp)
    except Exception as e:
        print(f"Error saving results: {e}")

    print(f"\nEvaluation completed! Results saved to: {output_dir}")

    # Return results for further analysis if needed
    return {
        'all_metrics': all_metrics,
        'pattern_metrics': pattern_metrics,
        'results': results,
        'output_dir': output_dir,
        'timestamp': timestamp
    }

# 9. Additional Analysis Functions

def analyze_error_cases(results, all_metrics, top_n=10):
    """
    Analyze the most problematic cases (highest errors)
    """
    print(f"\nANALYZING TOP {top_n} ERROR CASES:")
    print("="*60)

    # Find false positives (different speakers with high similarity)
    false_positives = [(r, r['similarity']) for r in results
                      if not r['is_same_speaker'] and r['similarity'] >= all_metrics['eer_threshold']]
    false_positives.sort(key=lambda x: x[1], reverse=True)

    print(f"\nTOP {min(top_n, len(false_positives))} FALSE POSITIVES (Different speakers with high similarity):")
    for i, (r, score) in enumerate(false_positives[:top_n]):
        print(f"{i+1}. Score: {score:.4f} | Speakers: {r['speaker1']} vs {r['speaker2']} | "
              f"Pattern: {r['pattern']} | Files: {r['file1']}, {r['file2']}")

    # Find false negatives (same speakers with low similarity)
    false_negatives = [(r, r['similarity']) for r in results
                      if r['is_same_speaker'] and r['similarity'] < all_metrics['eer_threshold']]
    false_negatives.sort(key=lambda x: x[1])

    print(f"\nTOP {min(top_n, len(false_negatives))} FALSE NEGATIVES (Same speakers with low similarity):")
    for i, (r, score) in enumerate(false_negatives[:top_n]):
        print(f"{i+1}. Score: {score:.4f} | Speaker: {r['speaker1']} | "
              f"Pattern: {r['pattern']} | Files: {r['file1']}, {r['file2']}")

def cross_pattern_analysis(results):
    """
    Analyze performance across different speaking style combinations
    """
    print("\nCROSS-PATTERN ANALYSIS:")
    print("="*50)

    # Create pattern confusion matrix
    patterns = ['cl', 'ch', 'rd']  # clean, child, read
    pattern_matrix = np.zeros((len(patterns), len(patterns)))
    pattern_counts = np.zeros((len(patterns), len(patterns)))

    pattern_to_idx = {p: i for i, p in enumerate(patterns)}

    for r in results:
        if r['style1'] in pattern_to_idx and r['style2'] in pattern_to_idx:
            i, j = pattern_to_idx[r['style1']], pattern_to_idx[r['style2']]
            pattern_matrix[i, j] += r['similarity']
            pattern_counts[i, j] += 1

    # Average similarities
    with np.errstate(divide='ignore', invalid='ignore'):
        avg_pattern_matrix = np.divide(pattern_matrix, pattern_counts,
                                      out=np.zeros_like(pattern_matrix),
                                      where=pattern_counts!=0)

    print("Average Similarity Matrix (rows: enrollment, cols: test):")
    print(f"{'':>8}", end="")
    for p in patterns:
        print(f"{p:>8}", end="")
    print()

    for i, p1 in enumerate(patterns):
        print(f"{p1:>8}", end="")
        for j, p2 in enumerate(patterns):
            if pattern_counts[i, j] > 0:
                print(f"{avg_pattern_matrix[i, j]:>8.3f}", end="")
            else:
                print(f"{'N/A':>8}", end="")
        print()

# Run the evaluation
if __name__ == "__main__":
    # Execute main evaluation
    evaluation_results = main_evaluation()

    # Additional analysis if evaluation was successful
    if evaluation_results:
        print("\n" + "="*80)
        print("ADDITIONAL ANALYSIS")
        print("="*80)

        # Error case analysis
        analyze_error_cases(evaluation_results['results'],
                          evaluation_results['all_metrics'],
                          top_n=5)

        # Cross-pattern analysis
        cross_pattern_analysis(evaluation_results['results'])

        print("\n" + "="*80)
        print("EVALUATION COMPLETE")
        print("="*80)
        print(f"All results and plots saved to: {evaluation_results['output_dir']}")

Mounted at /content/drive
Using device: cpu
Starting X-Vector Speaker Verification Evaluation...
Loading trained X-Vector model...
Loaded model from epoch 49
Model loaded successfully!

Processing verification trials...


Processing Verification Trials: 100%|██████████| 4800/4800 [50:18<00:00,  1.59it/s]
/usr/local/lib/python3.11/dist-packages/numpy/_core/function_base.py:168: RuntimeWarning: invalid value encountered in multiply
  y *= step
/usr/local/lib/python3.11/dist-packages/scipy/interpolate/_interpolate.py:482: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
/usr/local/lib/python3.11/dist-packages/numpy/_core/function_base.py:168: RuntimeWarning: invalid value encountered in multiply
  y *= step
/usr/local/lib/python3.11/dist-packages/scipy/interpolate/_interpolate.py:482: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
/usr/local/lib/python3.11/dist-packages/numpy/_core/function_base.py:168: RuntimeWarning: invalid value encountered in multiply
  y *= step
/usr/local/lib/python3.11/dist-packages/scipy/interpolate/_interpolate.py:482: RuntimeWarning: invalid value encountered in multiply
  y

Processed 4800 verification trials

Computing performance metrics...
y_true shape: (4800,), unique values: (array([0, 1]), array([2400, 2400]))
y_score shape: (4800,), range: [-0.47967952489852905, 0.973887026309967]
EER computation - labels shape: (4800,), unique values: (array([0, 1]), array([2400, 2400]))
EER computation - scores shape: (4800,), range: [-0.47967952489852905, 0.973887026309967]
NaN values in interpolation, using direct method
Overall metrics computed successfully!

Analyzing performance by speaking style patterns...
y_true shape: (800,), unique values: (array([0, 1]), array([400, 400]))
y_score shape: (800,), range: [-0.33619439601898193, 0.973887026309967]
EER computation - labels shape: (800,), unique values: (array([0, 1]), array([400, 400]))
EER computation - scores shape: (800,), range: [-0.33619439601898193, 0.973887026309967]
NaN values in interpolation, using direct method
y_true shape: (400,), unique values: (array([0, 1]), array([200, 200]))
y_score shape: 

/usr/local/lib/python3.11/dist-packages/numpy/_core/function_base.py:168: RuntimeWarning: invalid value encountered in multiply
  y *= step
/usr/local/lib/python3.11/dist-packages/scipy/interpolate/_interpolate.py:482: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
/usr/local/lib/python3.11/dist-packages/numpy/_core/function_base.py:168: RuntimeWarning: invalid value encountered in multiply
  y *= step
/usr/local/lib/python3.11/dist-packages/scipy/interpolate/_interpolate.py:482: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
/usr/local/lib/python3.11/dist-packages/numpy/_core/function_base.py:168: RuntimeWarning: invalid value encountered in multiply
  y *= step
/usr/local/lib/python3.11/dist-packages/scipy/interpolate/_interpolate.py:482: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
/usr/local/lib/python3.11/dist-packages/

NaN values in interpolation, using direct method
y_true shape: (800,), unique values: (array([0, 1]), array([400, 400]))
y_score shape: (800,), range: [-0.40662527084350586, 0.9620280265808105]
EER computation - labels shape: (800,), unique values: (array([0, 1]), array([400, 400]))
EER computation - scores shape: (800,), range: [-0.40662527084350586, 0.9620280265808105]
NaN values in interpolation, using direct method
Analyzed 9 patterns

Creating visualization plots...
Creating comprehensive visualization plots...
Visualization plots created successfully!

X-VECTOR SPEAKER VERIFICATION EVALUATION RESULTS

OVERALL STATISTICS:
Total trials: 4800
Same speaker trials: 2400
Different speaker trials: 2400

OVERALL PERFORMANCE METRICS:
Equal Error Rate (EER): 0.2992 (29.92%)
EER Threshold: 0.5613
Area Under Curve (AUC): 0.7669
Average Precision (AP): 0.7670
Accuracy at EER threshold: 0.7008
Detection Cost Function (DCF) @ p_target=0.01: 0.0099
Detection Cost Function (DCF) @ p_target=0.5: 0